# 导入依赖和文件

In [10]:
import json
import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

In [11]:

# 加载特征数据
X = np.load("X.npy")
X_test = np.load("X_test.npy")
y = np.load("y.npy")

# 划分训练集与验证集

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, train_size=0.8, test_size=0.2, random_state=0)


# 寻找并保存最优参数

In [12]:

# 定义 LGBM 的目标函数
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 6),
        'num_leaves': trial.suggest_int('num_leaves', 20, 30),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 定义 CatBoost 的目标函数
def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 6),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 3),
        'border_count': trial.suggest_categorical('border_count', [32, 64]),
        'verbose': False,
        'random_state': 0
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 创建 Optuna 研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=50)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=50)

# 获取最优参数
lgbm_best_params = lgbm_study.best_params
lgbm_best_value = lgbm_study.best_value
catboost_best_params = catboost_study.best_params
catboost_best_value = catboost_study.best_value

[I 2025-04-27 16:19:30,799] A new study created in memory with name: no-name-cf541202-283f-47cd-912c-fc06d297c08a
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:30,873] Trial 0 finished with value: 0.7855089131684876 and parameters: {'n_estimators': 100, 'learning_rate': 0.011734778914577892, 'max_depth': 6, 'num_leaves': 28, 'subsample': 0.9207487968054752, 'colsample_bytree': 0.8597511870606016}. Best is trial 0 with value: 0.7855089131684876.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:30,933] Trial 1 finished with value: 0.7780333525014376 and parameters: {'n_estimators': 50, 

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000316 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000797 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:31,034] Trial 4 finished with value: 0.7837837837837838 and parameters: {'n_estimators': 50, 'learning_rate': 0.03145081720919853, 'max_depth': 6, 'num_leaves': 20, 'subsample': 0.9045809872872947, 'colsample_bytree': 0.820481114571945}. Best is trial 0 with value: 0.7855089131684876.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:31,072] Trial 5 finished with value: 0.7855089131684876 and parameters: {'n_estimators': 100, 'learning_rate': 0.02479830627281829, 'max_depth': 4, 'num_leaves': 26, 'subsample': 0.8884593754947476, 'colsample_

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000265 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000267 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:31,239] Trial 9 finished with value: 0.7814836112708453 and parameters: {'n_estimators': 50, 'learning_rate': 0.03819631363838044, 'max_depth': 6, 'num_leaves': 24, 'subsample': 0.9804093237065422, 'colsample_bytree': 0.957868163447765}. Best is trial 7 with value: 0.7947096032202415.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:31,284] Trial 10 finished with value: 0.7975848188614146 and parameters: {'n_estimators': 100, 'learning_rate': 0.08149305382700582, 'max_depth': 4, 'num_leaves': 30, 'subsample': 0.9915927946097647, 'colsample

...

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:33,206] Trial 44 finished with value: 0.7843588269120184 and parameters: {'n_estimators': 100, 'learning_rate': 0.02098593232703624, 'max_depth': 6, 'num_leaves': 24, 'subsample': 0.922555464302864, 'colsample_bytree': 0.8131367341367859}. Best is trial 41 with value: 0.8067855089131685.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:33,267] Trial 45 finished with value: 0.7958596894767107 and parameters: {'n_estimators': 100, 'learning_rate': 0.05160381764282719, 'max_depth': 6, 'num_leaves': 23, 'subsample': 0.9043493279876355, 'colsam

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000292 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:33,432] Trial 48 finished with value: 0.7952846463484762 and parameters: {'n_estimators': 100, 'learning_rate': 0.060318425443663545, 'max_depth': 6, 'num_leaves': 20, 'subsample': 0.9297902161990709, 'colsample_bytree': 0.8463666511644555}. Best is trial 41 with value: 0.8067855089131685.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:33,493] Trial 49 finished with value: 0.7981598619896493 and parameters: {'n_estimators': 100, 'learning_rate': 0.09131448120115647, 'max_depth': 6, 'num_leaves': 23, 'subsample': 0.8823263541094234, 'cols

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000577 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-04-27 16:19:33,630] Trial 0 finished with value: 0.7918343875790684 and parameters: {'n_estimators': 50, 'learning_rate': 0.07896900522291145, 'depth': 4, 'l2_leaf_reg': 2, 'border_count': 32}. Best is trial 0 with value: 0.7918343875790684.
[I 2025-04-27 16:19:33,772] Trial 1 finished with value: 0.7912593444508338 and parameters: {'n_estimators': 50, 'learning_rate': 0.07732908278660322, 'depth': 6, 'l2_leaf_reg': 2, 'border_count': 32}. Best is trial 0 with value: 0.7918343875790684.
[I 2025-04-27 16:19:33,901] Trial 2 finished with value: 0.7849338700402531 and parameters: {'n_estimators': 50, 'learning_rate': 0.056630185989696026, 'depth': 4, 'l2_leaf_reg': 1, 'border_count': 64}. Best is trial 0 with value: 0.7918343875790684.
[I 2025-04-27 16:19:34,025] Trial 3 finished with value: 0.7699827487061529 and parameters: {'n_estimators': 50, 'learning_rate': 0.023350876179329298, 'depth': 4, 'l2_leaf_reg': 2, 'border_count': 64}. Best is trial 0 with value: 0.7918343875790684

In [13]:
print("LGBM 最佳得分:", lgbm_best_value)
print("CatBoost 最佳得分:", catboost_best_value)
# 导出 LGBM 和 CatBoost 的最优参数到 JSON 文件
with open('lgbm_best_params.json', 'w') as f:
    json.dump(lgbm_best_params, f)

with open('catboost_best_params.json', 'w') as f:
    json.dump(catboost_best_params, f)

LGBM 最佳得分: 0.8067855089131685
CatBoost 最佳得分: 0.7987349051178838


# 训练并保存模型

In [14]:

# 导入最优参数
with open('lgbm_best_params.json', 'r') as f:
    lgbm_best_params = json.load(f)

with open('catboost_best_params.json', 'r') as f:
    catboost_best_params = json.load(f)

# 使用最优参数重新实例化模型并训练
lgbm_model = LGBMClassifier(**lgbm_best_params, random_state=0)
catboost_model = CatBoostClassifier(**catboost_best_params, verbose=False, random_state=0)

lgbm_model.fit(X_train, y_train)
catboost_model.fit(X_train, y_train)

joblib.dump(lgbm_model, 'lgbm_best_model.joblib')
joblib.dump(catboost_model, 'catboost_best_model.joblib')


[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000543 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

['catboost_best_model.joblib']

# 模型融合与提交文件

基学习器已经训练好，现在使用元学习器进行模型融合

In [15]:
# # 加载训练好的基模型
# lgbm_model = joblib.load('lgbm_best_model.joblib')
# catboost_model = joblib.load('catboost_best_model.joblib')

# # 检查模型加载是否正确
# print(lgbm_model.get_params())
# print(catboost_model.get_params())

# # 定义生成元特征的函数
# def generate_meta_features(model, X_train, y_train, X_valid, y_valid, X_test, n_splits=5):
#     print("Entering generate_meta_features function")  # 调试信息
#     kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
#     meta_train = np.zeros((X_train.shape[0],))
#     meta_valid = np.zeros((X_valid.shape[0],))
#     meta_test = np.zeros((X_test.shape[0],))
    
#     for train_index, val_index in kf.split(X_train):
#         print("Inside KFold loop")  # 调试信息
#         X_tr, X_val = X_train[train_index], X_train[val_index]
#         y_tr = y_train[train_index]  # 获取当前折的训练目标变量
        
#         # 确保 X_tr 和 y_tr 的维度正确
#         print(f"X_tr shape: {X_tr.shape}, y_tr shape: {y_tr.shape}")
        
#         model.fit(X_tr, y_tr)  # 传递 y_tr 作为目标变量
#         meta_train[val_index] = model.predict_proba(X_val)[:, 1]
#         print("Completed a fold")  # 调试信息
    
#     # 使用整个训练集重新训练模型以生成验证集和测试集的元特征
#     print("Refitting model on entire training set")  # 调试信息
#     model.fit(X_train, y_train)
#     meta_valid = model.predict_proba(X_valid)[:, 1]
#     meta_test = model.predict_proba(X_test)[:, 1]
#     print("Exiting generate_meta_features function")  # 调试信息
    
#     return meta_train, meta_valid, meta_test

# # 为 LGBM 和 CatBoost 生成元特征
# print("Generating meta features for LGBM")  # 调试信息
# lgbm_meta_train, lgbm_meta_valid, lgbm_meta_test = generate_meta_features(lgbm_model, X_train, y_train, X_valid, y_valid, X_test)
# print("Generating meta features for CatBoost")  # 调试信息
# catboost_meta_train, catboost_meta_valid, catboost_meta_test = generate_meta_features(catboost_model, X_train, y_train, X_valid, y_valid, X_test)

# # 构建元特征矩阵
# X_train_meta = np.column_stack((lgbm_meta_train, catboost_meta_train))
# X_valid_meta = np.column_stack((lgbm_meta_valid, catboost_meta_valid))
# X_test_meta = np.column_stack((lgbm_meta_test, catboost_meta_test))

# # 定义元模型的目标函数
# def meta_objective(trial):
#     params = {
#         'C': trial.suggest_float('C', 0.01, 10.0, log=True),
#         'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']),
#         'random_state': 0
#     }
#     model = LogisticRegression(**params)
#     model.fit(X_train_meta, y_train)
#     y_pred = model.predict(X_valid_meta)
#     accuracy = accuracy_score(y_valid, y_pred)
#     print(f"Meta model accuracy: {accuracy:.4f}")  # 调试信息
#     return accuracy

# # 创建 Optuna 研究对象并进行优化
# print("Starting Optuna study for meta model")  # 调试信息
# meta_study = optuna.create_study(direction='maximize')
# meta_study.optimize(meta_objective, n_trials=50)

# # 获取最优参数
# meta_best_params = meta_study.best_params
# meta_best_value = meta_study.best_value

# print(f"元模型最优参数: {meta_best_params}")
# print(f"元模型最优准确率: {meta_best_value:.4f}")

# # 使用最优参数训练最终的元模型
# best_meta_model = LogisticRegression(**meta_best_params, random_state=0)
# best_meta_model.fit(X_train_meta, y_train)

# # 保存元模型
# joblib.dump(best_meta_model, 'best_meta_model.joblib')


In [16]:
# 加载训练好的基模型
lgbm_model = joblib.load('lgbm_best_model.joblib')
catboost_model = joblib.load('catboost_best_model.joblib')

# 检查模型加载是否正确
print("LGBM Model Parameters:", lgbm_model.get_params())
print("CatBoost Model Parameters:", catboost_model.get_params())

# 定义生成元特征的函数
def generate_meta_features(model, X_train, y_train, X_valid, y_valid, X_test, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    meta_train = np.zeros((X_train.shape[0],))
    meta_valid = np.zeros((X_valid.shape[0],))
    meta_test = np.zeros((X_test.shape[0],))
    
    for train_index, val_index in kf.split(X_train):
        X_tr, X_val = X_train[train_index], X_train[val_index]
        y_tr = y_train[train_index]
        model.fit(X_tr, y_tr)
        meta_train[val_index] = model.predict_proba(X_val)[:, 1]
    
    # 使用整个训练集重新训练模型以生成验证集和测试集的元特征
    model.fit(X_train, y_train)
    meta_valid = model.predict_proba(X_valid)[:, 1]
    meta_test = model.predict_proba(X_test)[:, 1]
    
    return meta_train, meta_valid, meta_test


print("Generating meta features for LGBM")
lgbm_meta_train, lgbm_meta_valid, lgbm_meta_test = generate_meta_features(lgbm_model, X_train, y_train, X_valid, y_valid, X_test)

print("Generating meta features for CatBoost")
catboost_meta_train, catboost_meta_valid, catboost_meta_test = generate_meta_features(catboost_model, X_train, y_train, X_valid, y_valid, X_test)

# 构建元特征矩阵
X_train_meta = np.column_stack((lgbm_meta_train, catboost_meta_train))
X_valid_meta = np.column_stack((lgbm_meta_valid, catboost_meta_valid))
X_test_meta = np.column_stack((lgbm_meta_test, catboost_meta_test))

# 定义元学习器的超参数优化目标函数
def logistic_regression_objective(trial):
    params = {
        'C': trial.suggest_float('C', 0.01, 10.0, log=True),
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']),
        'random_state': 0
    }
    model = LogisticRegression(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def random_forest_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': 0
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'random_state': 0
    }
    model = CatBoostClassifier(**params, verbose=0)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def lasso_objective(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 0.0001, 10.0, log=True),
        'random_state': 0
    }
    model = Lasso(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    y_pred_class = (y_pred > 0.5).astype(int)
    return accuracy_score(y_valid, y_pred_class)

# 创建Optuna研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=50)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=50)

logistic_regression_study = optuna.create_study(direction='maximize')
logistic_regression_study.optimize(logistic_regression_objective, n_trials=50)

random_forest_study = optuna.create_study(direction='maximize')
random_forest_study.optimize(random_forest_objective, n_trials=50)

lasso_study = optuna.create_study(direction='maximize')
lasso_study.optimize(lasso_objective, n_trials=50)

# 获取最优参数和准确率
meta_models = {
    'LogisticRegression': {
        'best_params': logistic_regression_study.best_params,
        'best_accuracy': logistic_regression_study.best_value
    },
    'RandomForest': {
        'best_params': random_forest_study.best_params,
        'best_accuracy': random_forest_study.best_value
    },
    'LGBMClassifier': {
        'best_params': lgbm_study.best_params,
        'best_accuracy': lgbm_study.best_value
    },
    'CatBoostClassifier': {
        'best_params': catboost_study.best_params,
        'best_accuracy': catboost_study.best_value
    },
    'LassoRegression': {
        'best_params': lasso_study.best_params,
        'best_accuracy': lasso_study.best_value
    }
}


# 选择最佳元学习器
best_meta_model_name = max(meta_models, key=lambda k: meta_models[k]['best_accuracy'])
best_meta_model_params = meta_models[best_meta_model_name]['best_params']
best_meta_model_accuracy = meta_models[best_meta_model_name]['best_accuracy']

# 根据最佳元学习器的名称创建模型实例并训练
if best_meta_model_name == 'LogisticRegression':
    best_meta_model = LogisticRegression(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'RandomForest':
    best_meta_model = RandomForestClassifier(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'LGBMClassifier':
    best_meta_model = LGBMClassifier(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'CatBoostClassifier':
    best_meta_model = CatBoostClassifier(**best_meta_model_params, random_state=0, verbose=0)
elif best_meta_model_name == 'LassoRegression':
    best_meta_model = Lasso(**best_meta_model_params, random_state=0)

# 如果是Lasso回归，需要将预测结果转换为分类标签
if best_meta_model_name == 'LassoRegression':
    best_meta_model.fit(X_train_meta, y_train)
    y_pred_final = best_meta_model.predict(X_valid_meta)
    y_pred_final_class = (y_pred_final > 0.5).astype(int)
    final_accuracy = accuracy_score(y_valid, y_pred_final_class)
    final_report = classification_report(y_valid, y_pred_final_class)
else:
    best_meta_model.fit(X_train_meta, y_train)
    y_pred_final = best_meta_model.predict(X_valid_meta)
    final_accuracy = accuracy_score(y_valid, y_pred_final)
    final_report = classification_report(y_valid, y_pred_final)

# 保存元模型
joblib.dump(best_meta_model, 'best_meta_model.joblib')



LGBM Model Parameters: {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.8351605780168891, 'importance_type': 'split', 'learning_rate': 0.08879763503223843, 'max_depth': 6, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 24, 'objective': None, 'random_state': 0, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.9496599763298335, 'subsample_for_bin': 200000, 'subsample_freq': 0}
CatBoost Model Parameters: {'learning_rate': 0.04293183210726391, 'depth': 6, 'l2_leaf_reg': 3, 'border_count': 32, 'verbose': False, 'n_estimators': 100, 'random_state': 0}
Generating meta features for LGBM
[LightGBM] [Info] Number of positive: 2802, number of negative: 2761
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000464 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:45,121] A new study created in memory with name: no-name-b902c456-da1d-46ae-90f6-10535ab5453a
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:45,164] Trial 0 finished with value: 0.7786083956296722 and parameters: {'n_estimators': 131, 'learning_rate': 0.2545904401722278, 'max_depth': 7, 'num_leaves': 95}. Best

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000095 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:48,583] Trial 47 finished with value: 0.7883841288096607 and parameters: {'n_estimators': 266, 'learning_rate': 0.02636020682647478, 'max_depth': 10, 'num_leaves': 88}. Best is trial 23 with value: 0.8033352501437608.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:19:48,656] Trial 48 finished with value: 0.7958596894767107 and parameters: {'n_estimators': 469, 'learning_rate': 0.03730074703923332, 'max_depth': 3, 'num_leaves': 72}. Best is trial 23 with value: 0.8033352501437608.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/pyth

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000088 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

[I 2025-04-27 16:19:49,155] Trial 0 finished with value: 0.7947096032202415 and parameters: {'n_estimators': 365, 'learning_rate': 0.14958344844987614, 'max_depth': 4}. Best is trial 0 with value: 0.7947096032202415.
[I 2025-04-27 16:19:49,614] Trial 1 finished with value: 0.7791834387579069 and parameters: {'n_estimators': 369, 'learning_rate': 0.20842614914949031, 'max_depth': 6}. Best is trial 0 with value: 0.7947096032202415.
[I 2025-04-27 16:19:49,699] Trial 2 finished with value: 0.8021851638872916 and parameters: {'n_estimators': 121, 'learning_rate': 0.07569653981704898, 'max_depth': 5}. Best is trial 2 with value: 0.8021851638872916.
[I 2025-04-27 16:19:50,260] Trial 3 finished with value: 0.7987349051178838 and parameters: {'n_estimators': 246, 'learning_rate': 0.017306751334161374, 'max_depth': 9}. Best is trial 2 with value: 0.8021851638872916.
[I 2025-04-27 16:19:50,876] Trial 4 finished with value: 0.7780333525014376 and parameters: {'n_estimators': 451, 'learning_rate': 

['best_meta_model.joblib']

In [17]:
for name, metrics in meta_models.items():
    print(f"{name}: Best Accuracy = {metrics['best_accuracy']:.4f}, Best Params = {metrics['best_params']}")
print(f"\nFinal Meta Model Evaluation:")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"Classification Report:\n{final_report}")

LogisticRegression: Best Accuracy = 0.7987, Best Params = {'C': 0.01067449790673757, 'solver': 'liblinear'}
RandomForest: Best Accuracy = 0.8039, Best Params = {'n_estimators': 283, 'max_depth': 4, 'min_samples_split': 8, 'min_samples_leaf': 8}
LGBMClassifier: Best Accuracy = 0.8033, Best Params = {'n_estimators': 362, 'learning_rate': 0.015442724924359918, 'max_depth': 4, 'num_leaves': 90}
CatBoostClassifier: Best Accuracy = 0.8068, Best Params = {'n_estimators': 135, 'learning_rate': 0.17073407844459842, 'max_depth': 4}
LassoRegression: Best Accuracy = 0.8091, Best Params = {'alpha': 0.08926875378086137}

Final Meta Model Evaluation:
Accuracy: 0.8091
Classification Report:
              precision    recall  f1-score   support

       False       0.82      0.79      0.80       863
        True       0.80      0.83      0.81       876

    accuracy                           0.81      1739
   macro avg       0.81      0.81      0.81      1739
weighted avg       0.81      0.81      0.81 

In [19]:
# 加载元模型
best_meta_model = joblib.load('best_meta_model.joblib')

# 生成预测结果
pred = best_meta_model.predict(X_test_meta)

# 加载测试数据
test_data = pd.read_csv('test.csv')

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Transported': pred > 0.5
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)